In [1]:
import os
import cv2
import numpy as np
import scipy.io
from scipy.spatial import KDTree
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Reshape
from tensorflow.keras.applications import VGG16
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.optimizers import Adam

# Check if GPU is detected
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [2]:

# input - image, output - image
def pad_to_multiple(image, factor=32):
    h, w = image.shape[:2]

    # Calculate how many pixels to add
    pad_h = (factor - (h % factor)) % factor
    pad_w = (factor - (w % factor)) % factor

    # Apply padding to the bottom and right only
    # (top, bottom, left, right)
    padded_img = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w,
                                    cv2.BORDER_CONSTANT, value=0)

    return padded_img


In [3]:
def get_base_model():
    base_model = VGG16(weights='imagenet', input_shape=(None, None, 3), include_top=False)

    # Start fully frozen — will unfreeze in phase 2 of training
    for layer in base_model.layers[:10]:
        layer.trainable = False

    block4_conv3 = base_model.get_layer("block4_conv3").output
    return tf.keras.Model(inputs=base_model.inputs, outputs=block4_conv3)

In [4]:
inputs = tf.keras.Input(shape=(None, None, 3))

x = get_base_model()(inputs) # Removed [0] to keep the batch dimension

init = RandomNormal(stddev=0.01)

x = Conv2D(512, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(512, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(512, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(256, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(128, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(1, (1, 1), activation='relu', dilation_rate=1, kernel_initializer=init, padding='same')(x)

# To remove the last dimension (channel) while keeping batch/height/width
out = x[..., 0]

model = tf.keras.Model(inputs=inputs, outputs=out)
model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional (Functional)         │ (None, None, None,     │     7,635,264 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, None, None,     │     1,179,904 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, None, None,     │       295,040 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, None, None, 64) │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, None, None, 1)  │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, None, None)     │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,263,489 (62.04 MB)

 Trainable params: 14,528,001 (55.42 MB)

 Non-trainable params: 1,735,488 (6.62 MB)

FOR INFERENCE ON VIDEO

In [10]:
import cv2
import numpy as np
import tensorflow as tf
import time
import io
from PIL import Image
from IPython.display import display
import ipywidgets as widgets
import matplotlib.cm as cm

# ── CONFIG ─────────────────────────────────────────────────────────────────
VIDEO_PATH        = '/content/Railway-platform-video.mp4'
FRAME_SKIP        = 5
INFER_WIDTH       = 640
INFER_HEIGHT      = 360
DISPLAY_VIDEO_W   = 360    # portrait video — width is smaller
DISPLAY_VIDEO_H   = 480    # portrait video — height is larger
DISPLAY_DENSITY_W = 360
DISPLAY_DENSITY_H = 480
# ──────────────────────────────────────────────────────────────────────────

model.load_weights('/content/CSRNet.weights.h5')

def predict_frame(frame_bgr):
    small     = cv2.resize(frame_bgr, (INFER_WIDTH, INFER_HEIGHT))
    frame_rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    im        = frame_rgb.astype(np.float32) / 255.0
    im[:,:,0] = (im[:,:,0] - np.mean(im[:,:,0])) / (np.std(im[:,:,0]) + 1e-8)
    im[:,:,1] = (im[:,:,1] - np.mean(im[:,:,1])) / (np.std(im[:,:,1]) + 1e-8)
    im[:,:,2] = (im[:,:,2] - np.mean(im[:,:,2])) / (np.std(im[:,:,2]) + 1e-8)
    padded    = pad_to_multiple(im, factor=32)
    X         = tf.constant(np.expand_dims(padded, 0), dtype=tf.float32)
    y_pred    = model(X, training=False)
    dmap      = y_pred[0].numpy()
    count     = max(0, round(float(tf.reduce_sum(dmap).numpy())))
    return count, dmap

def frame_to_jpeg(frame_bgr, target_w, target_h):
    resized = cv2.resize(frame_bgr, (target_w, target_h))
    rgb     = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    pil     = Image.fromarray(rgb)
    buf     = io.BytesIO()
    pil.save(buf, format='JPEG', quality=85)
    buf.seek(0)
    return buf.getvalue()

def density_to_jpeg(density_map, target_w, target_h):
    d_min, d_max = density_map.min(), density_map.max()
    norm         = (density_map - d_min) / (d_max - d_min + 1e-8)
    heatmap_rgb  = (cm.jet(norm)[:,:,:3] * 255).astype(np.uint8)
    heatmap_bgr  = cv2.cvtColor(heatmap_rgb, cv2.COLOR_RGB2BGR)
    resized      = cv2.resize(heatmap_bgr, (target_w, target_h))
    rgb          = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    pil          = Image.fromarray(rgb)
    buf          = io.BytesIO()
    pil.save(buf, format='JPEG', quality=85)
    buf.seek(0)
    return buf.getvalue()

# ── Video info ─────────────────────────────────────────────────────────────
cap          = cv2.VideoCapture(VIDEO_PATH)
video_fps    = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ORIG_W       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
ORIG_H       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration     = total_frames / video_fps
spf          = 1.0 / video_fps

print(f'Video     : {ORIG_W}x{ORIG_H}  {video_fps:.0f}FPS')
print(f'Duration  : {duration:.1f}s  ({total_frames} frames)')
print(f'Inference : {INFER_WIDTH}x{INFER_HEIGHT}  every {FRAME_SKIP} frames')
print(f'Display   : video={DISPLAY_VIDEO_W}x{DISPLAY_VIDEO_H}  '
      f'density={DISPLAY_DENSITY_W}x{DISPLAY_DENSITY_H}')
print()

# ── Widgets ────────────────────────────────────────────────────────────────
video_widget = widgets.Image(
    format='jpeg',
    width=DISPLAY_VIDEO_W,
    height=DISPLAY_VIDEO_H
)

density_widget = widgets.Image(
    format='jpeg',
    width=DISPLAY_DENSITY_W,
    height=DISPLAY_DENSITY_H
)

count_widget = widgets.HTML(
    value='<h1 style="font-size:36px; color:white; background:#111; '
          'padding:8px 20px; border-radius:8px; margin:4px; '
          'text-align:center;">Count: 0</h1>'
)

progress_widget = widgets.IntProgress(
    value=0, min=0, max=total_frames,
    description='',
    bar_style='info',
    layout=widgets.Layout(width='860px', height='16px')
)

info_widget = widgets.Label(
    value='Starting...',
    layout=widgets.Layout(width='860px')
)

# Labels above each panel
video_label   = widgets.HTML('<b style="font-size:13px">Original Video</b>')
density_label = widgets.HTML('<b style="font-size:13px">Density Map</b>')

video_col   = widgets.VBox([video_label,   video_widget],
                            layout=widgets.Layout(align_items='center', margin='0 10px'))
density_col = widgets.VBox([density_label, density_widget],
                            layout=widgets.Layout(align_items='center', margin='0 10px'))

panels = widgets.HBox(
    [video_col, density_col],
    layout=widgets.Layout(align_items='flex-start', justify_content='center')
)

ui = widgets.VBox(
    [panels, count_widget, progress_widget, info_widget],
    layout=widgets.Layout(align_items='center', width='900px')
)

display(ui)   # displayed ONCE

# ── Main loop ──────────────────────────────────────────────────────────────
frame_idx  = 0
last_count = 0
last_dmap  = np.zeros((INFER_HEIGHT // 8, INFER_WIDTH // 8))
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ── Inference every FRAME_SKIP frames ─────────────────────────────
    if frame_idx % FRAME_SKIP == 0:
        last_count, last_dmap = predict_frame(frame)

        # Update density and count only when inference runs
        density_widget.value = density_to_jpeg(last_dmap,
                                               DISPLAY_DENSITY_W,
                                               DISPLAY_DENSITY_H)
        count_widget.value = (
            f'<h1 style="font-size:36px; color:white; background:#111; '
            f'padding:8px 20px; border-radius:8px; margin:4px; '
            f'text-align:center;">Count: {last_count}</h1>'
        )

    # ── Always update video frame ──────────────────────────────────────
    video_widget.value = frame_to_jpeg(frame, DISPLAY_VIDEO_W, DISPLAY_VIDEO_H)

    # ── Progress and info ──────────────────────────────────────────────
    progress_widget.value = frame_idx
    real_video_time       = frame_idx / video_fps
    elapsed               = time.time() - start_time
    info_widget.value     = (
        f'Frame {frame_idx+1}/{total_frames}  |  '
        f'Count: {last_count}  |  '
        f'Video time: {real_video_time:.1f}s / {duration:.1f}s  |  '
        f'Elapsed: {elapsed:.1f}s'
    )

    # ── Drift-corrected throttle to match original video speed ─────────
    target_elapsed_total = (frame_idx + 1) * spf
    actual_elapsed_total = time.time() - start_time
    sleep_time           = max(0.0, target_elapsed_total - actual_elapsed_total)
    if sleep_time > 0:
        time.sleep(sleep_time)

    frame_idx += 1

cap.release()
progress_widget.value = total_frames
info_widget.value     = (f'✅ Done — {frame_idx} frames  |  '
                          f'Duration: {duration:.1f}s  |  '
                          f'Total elapsed: {time.time()-start_time:.1f}s')

Video     : 1080x1920  60FPS
Duration  : 15.4s  (924 frames)
Inference : 640x360  every 5 frames
Display   : video=360x480  density=360x480

